In [3]:
from discovery_child_development.getters import crunchbase
import pandas as pd

import datetime

import importlib
importlib.reload(crunchbase);

from discovery_child_development import PROJECT_DIR
ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [15]:
# organisation data
organizations_df = crunchbase.get_cb_from_s3(table='organizations')
# description texts
organization_descriptions = crunchbase.get_cb_from_s3(table='organization_descriptions')

In [23]:
# select companies in the "correct" industries or with the keywords
all_categories = [c.split(",") for c in organizations_df.category_list.dropna().to_list()]
all_categories = [item for sublist in all_categories for item in sublist]

In [26]:
all_categories = sorted(list(set(all_categories)))

In [28]:
relevant_categories = [
    "Education",
    "Child care",
    "Parenting",
    "Children",
    "Toys",
    "Baby",
    "Underserved Children",
    "Family",
    "Charter Schools",
    "Vocational Education",
    "Continuing Education",
    "EdTech",
    "Music Education",
]
len(relevant_categories)

13

In [29]:
# check if relevant category strings present
orgs_category_df = (
    organizations_df
    .dropna(subset=['category_list'])
    .astype({'category_list': str})
    .assign(has_category = lambda df: df.category_list.str.contains('|'.join(relevant_categories)))
    .query('has_category')
)

In [39]:
orgs_descriptions_df = (
    organizations_df[['id', 'name', 'short_description']]
    .merge(organization_descriptions[['id', 'description']], on='id', how='left')
    .assign(text = lambda df: df.short_description.fillna('') + '. ' + df.description.fillna(''))
)[['id', 'text']]

In [46]:
# get keywords
from discovery_child_development.utils import keywords as kw
from discovery_child_development import PROJECT_DIR
KEYWORD_FILE = PROJECT_DIR / "discovery_child_development/config/patents/keywords_query.txt"
keywords = kw.get_keywords(KEYWORD_FILE)
simple_keywords = ['child', 'infant', 'baby', 'prenatal', 'pregnancy', 'pregnant', 'toddler', 'family', 'parent']
simple_keywords = [[k] for k in simple_keywords]
keywords = keywords + simple_keywords

In [64]:
orgs_descriptions_df_ = (
    orgs_descriptions_df
    .assign(has_keywords = lambda df: kw.check_keyword_hits(df.text, keywords))
    .query('has_keywords')
)

In [75]:
len(orgs_descriptions_df), len(orgs_descriptions_df_)

(3431820, 124480)

In [78]:
orgs_descriptions_df_.sample(10)

,id,text,has_keywords
92058,b661e625-b9a9-f545-868a-8bca21fb63eb,Help We've Got Kids is a local online (and for...,True
221415,d00fba8f-6775-4927-3585-6ba6f7deaaa5,"DSI Toys designs, develops, markets and distri...",True
363006,0fe15f33-d644-e05f-0939-b3897a4cdcee,Matchbook combines the best in both public sch...,True
214271,ed1580b4-b5a9-a480-5fd0-607fbd93b90b,Kashmir Luxury Hair is a family owned and oper...,True
1760251,ec952ddf-fe32-492f-aaa5-f2b160a286ca,Koseikai provides individual and family social...,True
12010,25602213-9636-5cd8-1567-910e7a6868bb,Rainfall of Envelopes makes it easy for indivi...,True
245762,198edcec-9f5d-33d8-4773-f59ddb182b71,Shambhala Music Festival was born from a visio...,True
1995053,e92454b0-92ab-4d99-9601-129c7d6b1a94,"Premiere Pediatrics provides newborn care, sic...",True
3156182,ff5972d2-a723-4783-810a-141232dd35ff,Papapá offers natural food products for babies...,True
3110814,c4cd3993-bb7e-460b-8b8e-da11dcfc1369,Mossbourne Community Academy is a provide the ...,True


In [65]:
len(orgs_descriptions_df_), len(orgs_category_df)

(124480, 139008)

In [83]:
orgs_category_df_ = orgs_category_df.merge(orgs_descriptions_df[['id', 'text']], on='id', how='left')[['id', 'text']]

In [86]:
len(orgs_category_df_)

139008

In [87]:
orgs_texts = (
    pd.concat([orgs_descriptions_df_[['id', 'text']], orgs_category_df_[['id', 'text']]])
    .drop_duplicates(subset='id')
    .dropna(subset=['text'])
)

In [91]:
len(orgs_texts)

240651

In [92]:
orgs_texts.head()

,id,text
9,4111dc8b-c0df-2d24-ed33-30cd137b3098,Geni is an online community of genealogists co...
34,5429709f-5e15-bacb-76d5-73c5ce00f938,To be able to contribute to society and to hel...
86,8a899f9c-886e-2b9c-378e-49bbeb316cfc,Topix is a news community connecting people to...
91,1029072c-c37e-5a60-801f-dfa5ce1fc337,REV is a financially focused corporate VC back...
104,cfe3ce79-49e0-77e4-7d45-7c75b902ef04,PopSugar is an online network of websites that...


In [93]:
orgs_texts.to_csv(ENRICHED_DATA_DIR / 'crunchbase_texts.csv', index=False)

## Check relevant gtr texts

In [94]:
from nesta_ds_utils.loading_saving import S3
from discovery_child_development import S3_BUCKET

In [96]:
org_texts_relevant_df = S3.download_obj(S3_BUCKET, f"data/outputs/binary_classifier/crunchbase_texts_relevance_labelled.csv", "dataframe")

In [97]:
org_texts_relevant_df.head(1)

,Unnamed: 0,id,text,predictions
0,0,4111dc8b-c0df-2d24-ed33-30cd137b3098,Geni is an online community of genealogists co...,0


In [99]:
len(org_texts_relevant_df.query("predictions == 1"))

8696

In [100]:
pd.set_option("max_colwidth", 1000)
org_texts_relevant_df.query("predictions == 1").to_csv(ENRICHED_DATA_DIR / 'crunchbase_texts_relevant.csv', index=False)